# AgentMeter — Pilot (Google Colab, free T4)

Per-agent resource benchmarking of one open-source LLM in a **minimal linear**
Perceive → Reason → Decide → Act network-threat-detection pipeline.

Dataset: `data/cicids_pilot.csv` (real CIC-IDS flows, 78 features). Taxonomy is
**5 classes**: Brute Force, Volumetric DDoS, Port Scanning, DoS Hulk, Benign.

**Tahap 1 guardrails (unchanged):**
- Pipeline is the *test subject* only — minimal, generic, linear. No governance,
  correlation, alerting, or retry/self-correction loops.
- **Sequential** execution only (never parallel — it contaminates readings).
- **Mode A**: each model is pulled from the HF Hub and run in-process on the GPU,
  so per-agent VRAM is measurable via `torch.cuda` / `pynvml`.
- **Subprocess-per-model VRAM isolation**: each model runs in its **own worker
  subprocess** (`agentmeter.worker`) that loads exactly one model, measures its
  fresh-context baseline, runs the scenarios, then **exits** — so the OS reclaims
  all GPU memory and the next model starts from a clean CUDA context. This
  reliably frees bitsandbytes 4-bit / `device_map` weights that in-process unload
  could not. Each model's `vram_guard` verifies it *started* clean.
- All config in `config.yaml` / the pilot config. Nothing hard-coded.
- GPU required: if `torch.cuda` is unavailable the pilot **STOPS** (no CPU fallback).

### ⚠️ Quantization notice (declare in your thesis)
A 7–8B model does **not** fit in fp16 on a 15 GB T4. This pilot uses **4-bit NF4**
quantization (bitsandbytes). The VRAM / latency / token numbers are therefore for
the **quantized** model, and any model-vs-model comparison must use the **same**
quantization to stay fair. This is a methodological choice you must state.

### Cost
Colab free tier has no \$ cost, but sessions are time-limited and the GPU can be
reclaimed. Save the result JSON files as soon as the run finishes.

## 1. Confirm you have a GPU runtime
Runtime → Change runtime type → Hardware accelerator = **T4 GPU**. Then run:

In [1]:
!nvidia-smi -L || echo 'NO GPU: set Runtime -> Change runtime type -> T4 GPU'

GPU 0: Tesla T4 (UUID: GPU-a136d1f7-9a95-f40f-788f-4092b49230fd)


## 2. Get the AgentMeter code
Clones the repo and **hard-resets** the working tree to `origin/BRANCH`, so a
stale clone from an earlier session can't linger. If the repo is **private**,
paste a GitHub token when prompted (input is hidden); if public, press Enter.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/cool-ride-mitmzl'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('AgentMeter'):
    subprocess.run(['git', 'clone', url + '.git', 'AgentMeter'], check=True)
os.chdir('AgentMeter')
# Force EXACTLY origin/BRANCH — a stale clone from a prior run can't linger.
subprocess.run(['git', 'remote', 'set-url', 'origin', url + '.git'], check=True)
subprocess.run(['git', 'fetch', '--force', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('cwd:', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())
print('HEAD :', subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

## 3. Install dependencies
Colab already ships a CUDA build of `torch` — we do **not** reinstall it (that
would risk breaking CUDA). We add the CPU-set deps plus the GPU/HF extras and
`bitsandbytes` for 4-bit.

In [ ]:
# CPU-set deps (langgraph, pandas, numpy, scipy, pyyaml) — no torch here
!pip install -q -r requirements.txt
# GPU / HF extras (torch already present on Colab)
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — switch runtime to T4 before continuing.'

## 4. Authenticate with your Hugging Face token (secure)
Entered via `getpass` — **not** hardcoded, not printed, not saved to the notebook.
You must have accepted the model's gated licence on its Hub page first.

Alternatively use a Colab Secret named `HF_TOKEN` (🔑 panel) — the cell picks it up.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 5. Lock the run settings (shared by both batches)
The 5 models are split into **two disk-safe batches** because 5 sets of weights
overflow Colab's disk. The split is **only** to manage disk — both batches use
the **same** `CONFIG`, the **same** 4-bit quant, the **same** scenarios and must
run on the **same** GPU, or the 5-model comparison is invalid. `SESSION_GPU` is
captured here and re-checked before Batch 2.

In [ ]:
import torch
CONFIG = 'configs/pilot_colab_t4.yaml'
N = 1   # quick isolation check; raise to the real N (e.g. 10) for the pilot

# Split ONLY to fit Colab disk — the two batches stay fully comparable.
BATCH1 = ['mistralai/Mistral-7B-Instruct-v0.3',
          'meta-llama/Meta-Llama-3-8B-Instruct',
          'Qwen/Qwen2.5-7B-Instruct']
BATCH2 = ['microsoft/Phi-3-mini-4k-instruct',
          'google/gemma-2-9b-it']

assert torch.cuda.is_available(), 'No CUDA GPU — switch runtime to T4 before continuing.'
SESSION_GPU = torch.cuda.get_device_name(0)
print('Session GPU  :', SESSION_GPU)
print('CONFIG       :', CONFIG, '| N per model:', N)
print('Batch 1 (3)  :', BATCH1)
print('Batch 2 (2)  :', BATCH2)

## 6. Batch 1 (3 models)
Runs the first three models, each in its own worker subprocess (sequential;
clean fresh CUDA context per model). Writes `results/pilot_<model>.json` for each
plus `results/pilot_combined.json`.

> If a model errors (e.g. gated licence not accepted, or a download 401), its
> worker exits non-zero and `run_pilot_models` **aborts the batch**. Fix that one
> model, then re-run **this cell** before moving on.

In [ ]:
from agentmeter.pilot import run_pilot_models
run_pilot_models(models=BATCH1, config_path=CONFIG, n=N)

## 7. Download Batch 1 results BEFORE clearing disk
Saves every `results/pilot_*.json` produced so far — download to your computer,
and (if Google Drive is already mounted) copy to Drive as a backup so the results
survive if the session drops. Also prints the isolation check for this batch.

In [ ]:
import glob, json, os, shutil
from google.colab import files

def _isolation(combined='results/pilot_combined.json'):
    try:
        c = json.load(open(combined))
    except Exception as e:
        print('  (no combined file to summarize:', e, ')'); return
    print('  fresh-context baseline (MB):', c.get('baseline_vram_mb'))
    for m in c.get('models', []):
        g = m.get('vram_guard') or {}
        print(f"    {str(m.get('model_label')):<40} before_load={m.get('device_vram_before_load_mb')}"
              f" guard.exceeded={g.get('exceeded')} in_process_residual_mb={m.get('in_process_residual_mb')}")

saved = sorted(glob.glob('results/pilot_*.json'))
print('Batch 1 result files:')
for f in saved: print('  -', f)
_isolation()

drive_dir = '/content/drive/MyDrive'
if os.path.isdir(drive_dir):
    try:
        dst = os.path.join(drive_dir, 'agentmeter_results'); os.makedirs(dst, exist_ok=True)
        for f in saved: shutil.copy(f, dst)
        print('Backed up to', dst)
    except Exception as e:
        print('Drive copy skipped:', e)
else:
    print('Drive not mounted — skipping Drive backup (mount it first if you want one).')

for f in saved:
    print('downloading', f)
    try:
        files.download(f)
    except Exception as e:
        print('  download manually from the Files panel:', e)

## 8. Clear Batch 1 model caches to free disk
Deletes **only** the Hugging Face **model** caches for the Batch 1 repo ids —
never datasets, never system files, never the Batch 2 models. This frees the disk
that 3 models' fp16 weights consumed so Batch 2 can download.

In [ ]:
import shutil
from huggingface_hub import scan_cache_dir

def _free_gb():
    return shutil.disk_usage('/').free / 1e9

print(f'Disk free before: {_free_gb():.1f} GB')
cache = scan_cache_dir()
batch1 = set(BATCH1)
to_delete = []
for repo in cache.repos:
    if repo.repo_type == 'model' and repo.repo_id in batch1:   # models only, Batch 1 only
        to_delete += [rev.commit_hash for rev in repo.revisions]
if to_delete:
    strategy = cache.delete_revisions(*to_delete)
    print(f'Deleting {len(to_delete)} revision(s) (~{strategy.expected_freed_size/1e9:.1f} GB) for:')
    for r in sorted(batch1): print('  -', r)
    strategy.execute()
else:
    print('No Batch 1 model caches found to delete.')
print(f'Disk free after : {_free_gb():.1f} GB')

## 9. Batch 2 (2 models)
Runs the remaining two models. First it **re-checks the GPU**: if Colab handed you
a different GPU than Batch 1 ran on, latency/VRAM are not comparable and the cell
stops loudly — re-run all 5 models on one GPU. `run_pilot_models` overwrites
`results/pilot_combined.json` with the Batch 2 combined, but the Batch 1 per-model
JSONs you already downloaded are unaffected.

In [ ]:
import torch
now_gpu = torch.cuda.get_device_name(0)
if now_gpu != SESSION_GPU:
    print('!' * 70)
    print(f'GPU CHANGED between batches: was {SESSION_GPU!r}, now {now_gpu!r}.')
    print('Latency/VRAM are NOT comparable across GPUs.')
    print('Re-run ALL 5 models on ONE GPU (restart the session and start over).')
    print('!' * 70)
assert now_gpu == SESSION_GPU, f'GPU changed ({SESSION_GPU} -> {now_gpu}); batches not comparable.'

from agentmeter.pilot import run_pilot_models
run_pilot_models(models=BATCH2, config_path=CONFIG, n=N)

## 10. Download Batch 2 results
Saves the Batch 2 per-model JSONs and the (now Batch 2) combined file, with the
same Drive-backup + isolation check as Batch 1. You now have all 5 per-model
`pilot_<model>.json` files locally for the side-by-side comparison.

In [ ]:
import json, os, shutil
from google.colab import files

def _isolation(combined='results/pilot_combined.json'):
    try:
        c = json.load(open(combined))
    except Exception as e:
        print('  (no combined file to summarize:', e, ')'); return
    print('  fresh-context baseline (MB):', c.get('baseline_vram_mb'))
    for m in c.get('models', []):
        g = m.get('vram_guard') or {}
        print(f"    {str(m.get('model_label')):<40} before_load={m.get('device_vram_before_load_mb')}"
              f" guard.exceeded={g.get('exceeded')} in_process_residual_mb={m.get('in_process_residual_mb')}")

# Batch 2 files only (the Batch 1 per-model JSONs are already downloaded).
saved = [f'results/pilot_{m.split("/")[-1]}.json' for m in BATCH2]
saved = [f for f in saved if os.path.exists(f)] + ['results/pilot_combined.json']
print('Batch 2 result files:')
for f in saved: print('  -', f)
_isolation()

drive_dir = '/content/drive/MyDrive'
if os.path.isdir(drive_dir):
    try:
        dst = os.path.join(drive_dir, 'agentmeter_results'); os.makedirs(dst, exist_ok=True)
        for f in saved: shutil.copy(f, dst)
        print('Backed up to', dst)
    except Exception as e:
        print('Drive copy skipped:', e)
else:
    print('Drive not mounted — skipping Drive backup.')

for f in saved:
    print('downloading', f)
    try:
        files.download(f)
    except Exception as e:
        print('  download manually from the Files panel:', e)

## 11. Per-model report (optional)
Render the single-model HTML report for each model produced so far (pure stdlib,
no extra installs). The side-by-side 5-model comparison is a separate local
Gradio demo that reads these saved JSON files.

In [ ]:
import glob
from IPython.display import HTML, display
for f in sorted(glob.glob('results/pilot_*.json')):
    if f.endswith('pilot_combined.json'): continue
    out = f.replace('.json', '_report.html')
    !python scripts/report.py "{f}" -o "{out}"
    display(HTML(open(out).read()))

## 12. Done — free the GPU
Runtime → Disconnect and delete runtime, so the free GPU is released for your next
session. Send back the five per-model `pilot_<model>.json` files (both batches)
for the side-by-side comparison demo.